# MRZ Extractor – Domino Data Lab

**Machine Readable Zone (MRZ) extraction and validation pipeline**  
Combines **PaddleOCR** (ONNX/HPI, CPU-only) with **OmniMRZ** for structured parsing.

---

## Prerequisites

1. **Domino Compute Environment** – Create or select an environment with the following installed:
   ```
   pip install paddlepaddle==2.6.1 paddleocr==2.8.1 onnxruntime==1.19.0 \
               omnimrz==0.1.6 opencv-python-headless==4.10.0.84 \
               pandas==2.2.2 openpyxl==3.1.5 Pillow==10.4.0
   ```
   Or attach the supplied `requirements.txt` to your environment's Dockerfile:
   ```dockerfile
   RUN pip install --no-cache-dir -r /app/requirements.txt
   ```

2. **Input images** – Place passport/ID card images in `/mnt/data/passports/`  
   (or update `INPUT_PATH` below).

3. **Output directory** – Results are written to `/mnt/artifacts/` by default.

---

## Notebook Structure

| Section | Description |
|---------|-------------|
| 1 | Configuration & paths |
| 2 | Import libraries |
| 3 | Initialise OCR engine |
| 4 | Image preprocessing helpers |
| 5 | MRZ extraction & OmniMRZ validation |
| 6 | Batch processing |
| 7 | Save JSON results |
| 8 | Convert JSON → Excel report |
| 9 | Quick summary |


## Section 1 – Configuration & Paths

Edit the variables in this cell to match your Domino workspace layout.

In [ ]:
import os

# ------------------------------------------------------------------ #
#  USER CONFIGURATION – adjust these paths for your Domino project   #
# ------------------------------------------------------------------ #

# Input: single image file OR directory containing images
INPUT_PATH: str = "/mnt/data/passports/"

# Output directory (created automatically if it does not exist)
OUTPUT_DIR: str = "/mnt/artifacts/"

# Output file names
OUTPUT_JSON: str  = os.path.join(OUTPUT_DIR, "mrz_results.json")
OUTPUT_EXCEL: str = os.path.join(OUTPUT_DIR, "mrz_report.xlsx")

# PaddleOCR settings
CPU_THREADS: int = os.cpu_count() or 4   # use all available cores
MAX_IMG_DIM: int = 2000                  # resize images larger than this

print(f"Input   : {INPUT_PATH}")
print(f"JSON out: {OUTPUT_JSON}")
print(f"XLSX out: {OUTPUT_EXCEL}")
print(f"Threads : {CPU_THREADS}")


## Section 2 – Import Libraries

In [ ]:
import json
import logging
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}

print("Libraries imported successfully.")


## Section 3 – Initialise the OCR Engine

PaddleOCR is configured with:
- `enable_hpi=True` → activates the ONNX Runtime / HPI backend for faster CPU inference
- `use_angle_cls=False` → skips the angle-classification model (MRZ is always horizontal)
- `use_gpu=False` → forces CPU execution
- Lightweight PP-OCRv4 *mobile* detection/recognition models

> **First run:** PaddleOCR will automatically download the pre-trained model weights (~20 MB).  
> Subsequent runs use the cached models.

In [ ]:
from paddleocr import PaddleOCR  # type: ignore

def init_ocr_engine(cpu_threads: int = CPU_THREADS) -> PaddleOCR:
    """
    Initialise PaddleOCR with CPU / HPI optimisations.

    Parameters
    ----------
    cpu_threads : int
        Number of CPU threads for inference.

    Returns
    -------
    PaddleOCR
        Configured OCR instance ready for inference.
    """
    logger.info("Initialising PaddleOCR (cpu_threads=%d) …", cpu_threads)
    engine = PaddleOCR(
        use_angle_cls=False,  # MRZ lines are always horizontal
        lang="en",
        use_gpu=False,
        enable_hpi=True,      # ONNX Runtime HPI backend
        cpu_threads=cpu_threads,
        show_log=False,
        # PP-OCRv4 mobile models – smallest / fastest
        det_model_dir=None,   # auto-download ch_PP-OCRv4_det_slim
        rec_model_dir=None,   # auto-download ch_PP-OCRv4_rec_slim
    )
    logger.info("PaddleOCR ready.")
    return engine


ocr_engine = init_ocr_engine()
print("OCR engine initialised.")


## Section 4 – Image Preprocessing Helpers

Good preprocessing significantly improves OCR accuracy on scanned/photographed documents:

1. **Resize** – Keeps the longest dimension within `MAX_IMG_DIM` to reduce memory and CPU load.
2. **CLAHE** – Contrast Limited Adaptive Histogram Equalisation improves legibility in poor-lighting or faded documents.

In [ ]:
def preprocess_image(
    image_path: str,
    max_dimension: int = MAX_IMG_DIM,
    apply_clahe: bool = True,
) -> np.ndarray:
    """
    Load and pre-process an image for optimal OCR on CPU.

    Parameters
    ----------
    image_path : str
        Path to the image file.
    max_dimension : int
        Maximum allowed edge length in pixels.
    apply_clahe : bool
        Apply CLAHE contrast enhancement if True.

    Returns
    -------
    np.ndarray
        Pre-processed BGR image array.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")

    # --- Resize if needed ------------------------------------------------
    h, w = img.shape[:2]
    scale = min(max_dimension / max(h, w), 1.0)
    if scale < 1.0:
        img = cv2.resize(
            img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA
        )

    # --- CLAHE contrast enhancement -------------------------------------
    if apply_clahe:
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    return img


print("Preprocessing helpers defined.")


## Section 5 – MRZ Extraction and OmniMRZ Validation

### 5a – PaddleOCR → candidate MRZ lines
OCR output is filtered to keep only strings that contain valid ICAO 9303 characters (`A-Z`, `0-9`, `<`) and are at least 30 characters long (minimum MRZ line width).

### 5b – OmniMRZ parsing
The candidate lines are passed to `MRZParser`, which handles:
- TD1 (3-line, 30-char) and TD3 (2-line, 44-char) formats
- Checksum verification for all mandatory fields
- Expiry date validation
- Structured field extraction (surname, given names, document number, nationality, dates …)

In [ ]:
from omnimrz import MRZParser  # type: ignore


def extract_mrz_lines(
    ocr: PaddleOCR,
    image: np.ndarray,
) -> List[str]:
    """
    Run PaddleOCR and return sanitised MRZ candidate lines.

    Parameters
    ----------
    ocr : PaddleOCR
        Initialised OCR engine.
    image : np.ndarray
        Pre-processed BGR image.

    Returns
    -------
    List[str]
        Filtered list of MRZ-like text strings.
    """
    raw_results = ocr.ocr(image, cls=False)
    if not raw_results or raw_results[0] is None:
        return []

    VALID_CHARS = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789<")
    candidates: List[str] = []

    for line in raw_results[0]:
        text: str = line[1][0].upper().replace(" ", "").replace("O", "0")
        clean = "".join(c for c in text if c in VALID_CHARS)
        if len(clean) >= 30:
            candidates.append(clean)

    return candidates


def parse_mrz(mrz_lines: List[str]) -> Dict[str, Any]:
    """
    Parse and validate MRZ lines using OmniMRZ.

    Parameters
    ----------
    mrz_lines : List[str]
        Candidate MRZ lines from OCR.

    Returns
    -------
    dict
        Structured MRZ fields and validation flags.
    """
    parser = MRZParser()
    raw_mrz = "\n".join(mrz_lines)
    try:
        result = parser.parse(raw_mrz)
        return result if isinstance(result, dict) else result.__dict__
    except Exception as exc:  # noqa: BLE001
        logger.warning("OmniMRZ parsing failed: %s", exc)
        return {"valid": False, "error": str(exc), "raw_lines": mrz_lines}


def process_image(image_path: str, ocr: PaddleOCR) -> Dict[str, Any]:
    """
    Full pipeline for a single image: preprocess → OCR → parse → validate.

    Parameters
    ----------
    image_path : str
        Path to the input image.
    ocr : PaddleOCR
        Initialised OCR engine.

    Returns
    -------
    dict
        Combined metadata + parsed MRZ fields.
    """
    t0 = time.perf_counter()
    record: Dict[str, Any] = {
        "file": Path(image_path).name,
        "file_path": image_path,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    try:
        img = preprocess_image(image_path)
        lines = extract_mrz_lines(ocr, img)
        record["raw_ocr_lines"] = lines
        if lines:
            mrz_data = parse_mrz(lines)
            record.update(mrz_data)
        else:
            record["valid"] = False
            record["error"] = "No MRZ candidates found by OCR."
    except Exception as exc:  # noqa: BLE001
        logger.error("Error on %s: %s", image_path, exc)
        record["valid"] = False
        record["error"] = str(exc)

    record["processing_time_seconds"] = round(time.perf_counter() - t0, 3)
    return record


print("Extraction and parsing functions defined.")


## Section 6 – Batch Processing

Collect all images from `INPUT_PATH` (single file or directory) and run the full pipeline.

In [ ]:
def collect_images(input_path: str) -> List[str]:
    """Return sorted list of image paths under input_path."""
    p = Path(input_path)
    if p.is_file():
        return [str(p.resolve())]
    return sorted(
        str(f.resolve())
        for f in p.rglob("*")
        if f.suffix.lower() in IMAGE_EXTENSIONS
    )


image_paths = collect_images(INPUT_PATH)
print(f"Found {len(image_paths)} image(s).")

all_results: List[Dict[str, Any]] = []
batch_start = time.perf_counter()

for i, img_path in enumerate(image_paths, 1):
    print(f"[{i}/{len(image_paths)}] Processing: {img_path}")
    result = process_image(img_path, ocr_engine)
    all_results.append(result)
    status = "✓ VALID" if result.get("valid") else "✗ INVALID"
    print(f"  → {status}  ({result['processing_time_seconds']}s)")

batch_elapsed = time.perf_counter() - batch_start
print(f"\nBatch complete: {len(all_results)} images in {batch_elapsed:.2f}s")


## Section 7 – Save Results as JSON

The raw results list (one dict per image) is serialised to a JSON file for archival and downstream processing.

In [ ]:
def save_json(results: List[Dict[str, Any]], path: str) -> None:
    """Serialise results to a JSON file."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(results, fh, indent=2, default=str)
    print(f"JSON saved → {path}")


save_json(all_results, OUTPUT_JSON)

# Quick preview of first result
if all_results:
    import pprint
    print("\n--- Preview: first result ---")
    pprint.pprint(all_results[0], depth=2)


## Section 8 – Convert JSON to Formatted Excel Report

Three sheets are created:

| Sheet | Contents |
|-------|----------|
| **Summary** | High-level status per image (valid, name, dates) |
| **Detailed** | All parsed MRZ fields |
| **Raw OCR** | Raw text lines from PaddleOCR for debugging |

Formatting applied: bold/coloured headers, auto column widths, frozen header row.

In [ ]:
DETAIL_COLUMNS: List[Tuple[str, str]] = [
    ("file",                    "File"),
    ("valid",                   "Valid"),
    ("document_type",           "Document Type"),
    ("country_code",            "Country Code"),
    ("surname",                 "Surname"),
    ("given_names",             "Given Names"),
    ("document_number",         "Document Number"),
    ("nationality",             "Nationality"),
    ("birth_date",              "Birth Date"),
    ("sex",                     "Sex"),
    ("expiry_date",             "Expiry Date"),
    ("personal_number",         "Personal Number"),
    ("checksum_valid",          "Checksum Valid"),
    ("is_expired",              "Is Expired"),
    ("error",                   "Error"),
    ("processing_time_seconds", "Processing Time (s)"),
    ("timestamp",               "Timestamp"),
]


def _header_style(ws) -> None:
    fill = PatternFill("solid", fgColor="1F4E79")
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = fill
        cell.alignment = Alignment(horizontal="center", vertical="center")


def _auto_width(ws) -> None:
    for col in ws.columns:
        max_len = max(
            len(str(c.value)) if c.value is not None else 0 for c in col
        )
        ws.column_dimensions[get_column_letter(col[0].column)].width = max(
            10, min(max_len + 2, 50)
        )


def build_excel(results: List[Dict[str, Any]], output_path: str) -> None:
    """
    Write a formatted three-sheet Excel workbook from results.

    Parameters
    ----------
    results : list
        List of per-image result dicts.
    output_path : str
        Destination .xlsx path.
    """
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    # -- Summary sheet --
    df_summary = pd.DataFrame([
        {
            "File":                r.get("file", ""),
            "Valid":               r.get("valid", False),
            "Document Type":       r.get("document_type", ""),
            "Surname":             r.get("surname", ""),
            "Given Names":         r.get("given_names", ""),
            "Nationality":         r.get("nationality", ""),
            "Birth Date":          r.get("birth_date", ""),
            "Expiry Date":         r.get("expiry_date", ""),
            "Is Expired":          r.get("is_expired", ""),
            "Checksum Valid":      r.get("checksum_valid", ""),
            "Error":               r.get("error", ""),
            "Processing Time (s)": r.get("processing_time_seconds", ""),
        }
        for r in results
    ])

    # -- Detailed sheet --
    df_detail = pd.DataFrame([
        {label: r.get(key, "") for key, label in DETAIL_COLUMNS}
        for r in results
    ])

    # -- Raw OCR sheet --
    df_ocr = pd.DataFrame([
        {
            "File":       r.get("file", ""),
            "OCR Line 1": (r.get("raw_ocr_lines") or [""])[0] if r.get("raw_ocr_lines") else "",
            "OCR Line 2": (r.get("raw_ocr_lines") or ["", ""])[1] if len(r.get("raw_ocr_lines") or []) > 1 else "",
            "OCR Line 3": (r.get("raw_ocr_lines") or ["", "", ""])[2] if len(r.get("raw_ocr_lines") or []) > 2 else "",
        }
        for r in results
    ])

    # Write sheets
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_summary.to_excel(writer, sheet_name="Summary",  index=False)
        df_detail.to_excel(writer,  sheet_name="Detailed", index=False)
        df_ocr.to_excel(writer,     sheet_name="Raw OCR",  index=False)

    # Apply formatting
    wb = load_workbook(output_path)
    for name in ["Summary", "Detailed", "Raw OCR"]:
        ws = wb[name]
        _header_style(ws)
        _auto_width(ws)
        ws.freeze_panes = "A2"
    wb.save(output_path)
    print(f"Excel saved → {output_path}")


build_excel(all_results, OUTPUT_EXCEL)


## Section 9 – Pipeline Summary

Display a quick overview of the batch results.

In [ ]:
total   = len(all_results)
valid   = sum(1 for r in all_results if r.get("valid"))
invalid = total - valid
avg_t   = sum(r.get("processing_time_seconds", 0) for r in all_results) / max(total, 1)

print("============================================================")
print("  MRZ Extraction Pipeline – Summary")
print("============================================================")
print(f"  Total images   : {total}")
print(f"  Valid MRZ      : {valid} ({100*valid//max(total,1)}%)")
print(f"  Invalid / Error: {invalid}")
print(f"  Avg time/image : {avg_t:.3f}s")
print(f"  JSON output    : {OUTPUT_JSON}")
print(f"  Excel output   : {OUTPUT_EXCEL}")
print("============================================================")

# Display the summary DataFrame for interactive review
import pandas as _pd
_df = _pd.read_excel(OUTPUT_EXCEL, sheet_name="Summary")
_df


---
## Example – Single Image Quick Test

Run the cell below to test the pipeline on **one image** without running the full batch.  
Replace the path with any passport/ID image in your Domino workspace.

In [ ]:
import pprint

SAMPLE_IMAGE = "/mnt/data/sample_passport.jpg"   # ← change to your image path

if Path(SAMPLE_IMAGE).exists():
    sample_result = process_image(SAMPLE_IMAGE, ocr_engine)
    print(f"Valid: {sample_result.get('valid')}")
    print(f"Name : {sample_result.get('surname')}, {sample_result.get('given_names')}")
    print(f"DOC# : {sample_result.get('document_number')}")
    print(f"DOB  : {sample_result.get('birth_date')}")
    print(f"EXP  : {sample_result.get('expiry_date')}  Expired={sample_result.get('is_expired')}")
    print(f"Time : {sample_result['processing_time_seconds']}s")
    print("\nFull result:")
    pprint.pprint(sample_result)
else:
    print(f"Sample image not found at: {SAMPLE_IMAGE}")
    print("Place a passport image there and re-run this cell.")
